# CascadeGuard - Test vs Trained (Colab Safe)

This notebook is submission-safe for Colab:

- It runs end-to-end without crashing if files are missing.
- It shows baseline-vs-trained and dashboard assets when available.
- It prints logs/artifacts when available.
- Heavy model loading is optional (disabled by default for reliability).

> ## Theory Box
> A trained adapter should beat baseline on the same split because:
> 1. SFT improves action-format reliability.
> 2. GRPO improves policy choices using environment rewards.
>
> Actual model stack used here:
> - Base model: `deepseek-ai/DeepSeek-R1-Distill-Qwen-32B`
> - Adapter tensors: `samarthdave0305/cascadeguard-trained`
> - Adapter tensors are loaded on top of the DeepSeek base model.
>
> Links:
> - Backend: https://harshit0400-backend.hf.space
> - Frontend: https://huggingface.co/spaces/LordBhatt/CascadeGuardUI
> - Env: https://huggingface.co/spaces/samarthdave0305/cascade-failure-env
> - Adapter: https://huggingface.co/samarthdave0305/cascadeguard-trained/tree/main
>
> Note: This notebook is designed to run safely even when large-model loading is skipped.

In [ ]:
import os
import sys
import json
import subprocess
import warnings
from pathlib import Path

from IPython.display import display, Markdown, Image

warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Samarth-Dave/cascade_gaurd_openEnv'
WORKDIR = Path('/content/cascade_guard' if IN_COLAB else '.').resolve()
RESULTS_DIR = WORKDIR / 'assets' / 'results'

LINKS = {
    'backend': 'https://harshit0400-backend.hf.space',
    'frontend': 'https://huggingface.co/spaces/LordBhatt/CascadeGuardUI',
    'env': 'https://huggingface.co/spaces/samarthdave0305/cascade-failure-env',
    'adapter': 'https://huggingface.co/samarthdave0305/cascadeguard-trained/tree/main',
}

print('IN_COLAB :', IN_COLAB)
print('WORKDIR  :', WORKDIR)
print('RESULTS  :', RESULTS_DIR)
print('Links:')
for k, v in LINKS.items():
    print(f'  - {k}: {v}')

In [ ]:
def run_cmd(cmd):
    try:
        print('>', ' '.join(cmd))
        out = subprocess.run(cmd, check=True, capture_output=True, text=True)
        if out.stdout.strip():
            print(out.stdout.strip()[:1000])
        return True
    except Exception as e:
        print('Command failed:', e)
        return False

if IN_COLAB:
    if not WORKDIR.exists():
        print('Cloning repository...')
        run_cmd(['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR)])
    else:
        print('Repository already exists, skipping clone.')
    os.chdir(WORKDIR)
else:
    print('Running outside Colab: skipping clone.')

print('Current directory:', Path('.').resolve())

In [ ]:
# Minimal optional installs for model-loading path; safe to skip if already available.
def ensure_import(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
        print(f'OK: {import_name}')
        return True
    except Exception:
        print(f'Installing: {pip_name}')
        ok = run_cmd([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
        if ok:
            try:
                __import__(import_name)
                print(f'Installed: {import_name}')
                return True
            except Exception as e:
                print(f'Import still failing for {import_name}:', e)
        return False

_ = ensure_import('pandas')
_ = ensure_import('PIL', 'pillow')
_ = ensure_import('matplotlib')

In [ ]:
EXPECTED = [
    ('training_loss_curve.png', 'SFT/GRPO loss over training step.'),
    ('training_reward_curve.png', 'GRPO reward over training step.'),
    ('baseline_vs_trained_reward.png', 'Baseline policy versus trained policy on the same evaluation split.'),
    ('demo_dashboard.png', 'Live CascadeGuard Space dashboard.'),
]

display(Markdown('## Requested Images'))
for name, caption in EXPECTED:
    p = RESULTS_DIR / name
    if p.exists():
        display(Markdown(f'### {name}'))
        display(Image(filename=str(p)))
        display(Markdown(f'Caption: {caption}'))
    else:
        display(Markdown(f'**Missing:** {p.as_posix()}'))

display(Markdown('## All Images Under assets/results'))
if RESULTS_DIR.exists():
    exts = {'.png', '.jpg', '.jpeg', '.webp'}
    all_images = sorted([x for x in RESULTS_DIR.rglob('*') if x.is_file() and x.suffix.lower() in exts])
    print(f'Found {len(all_images)} image(s).')
    for img in all_images:
        display(Markdown(f'- {img.relative_to(WORKDIR).as_posix()}'))
else:
    print('assets/results directory not found.')

In [ ]:
display(Markdown('## Logs and Evaluation Artifacts'))

candidate_files = [
    WORKDIR / 'cascadeguard_eval_results.csv',
    WORKDIR / 'baseline_results.json',
    WORKDIR / 'training_curves' / 'training_log.json',
    WORKDIR / 'cascadeguard_training_live.log',
    Path('/content/cascadeguard_training_live.log'),
]

for p in candidate_files:
    if not p.exists():
        continue
    print('\nFound:', p)
    try:
        if p.suffix.lower() == '.json':
            data = json.loads(p.read_text(encoding='utf-8', errors='ignore'))
            if isinstance(data, dict):
                print('JSON keys:', list(data.keys())[:20])
            elif isinstance(data, list):
                print('JSON list length:', len(data))
            else:
                print('JSON type:', type(data))
        elif p.suffix.lower() == '.csv':
            import pandas as pd
            df = pd.read_csv(p)
            display(df.tail(10))
        else:
            lines = p.read_text(encoding='utf-8', errors='ignore').splitlines()
            print('--- tail(40) ---')
            for line in lines[-40:]:
                print(line)
    except Exception as e:
        print('Could not read file:', e)

In [ ]:
# Optional heavy model load (disabled by default for error-free submission runs).
USE_HEAVY_MODEL = False

model = None
tokenizer = None

def normalize_hf_repo(value: str) -> str:
    if not value:
        return value
    v = value.strip()
    prefix = 'https://huggingface.co/'
    if v.startswith(prefix):
        v = v[len(prefix):]
    v = v.strip('/')
    if '/tree/' in v:
        v = v.split('/tree/')[0]
    return v

RAW_ADAPTER_REPO = 'https://huggingface.co/samarthdave0305/cascadeguard-trained/tree/main'
ADAPTER_REPO = normalize_hf_repo(RAW_ADAPTER_REPO)
BASE_MODEL = os.getenv('BASE_MODEL', 'deepseek-ai/DeepSeek-R1-Distill-Qwen-32B')

print('USE_HEAVY_MODEL:', USE_HEAVY_MODEL)
print('Adapter repo    :', ADAPTER_REPO)
print('Base model      :', BASE_MODEL)
print('CUDA available  :', __import__('torch').cuda.is_available())

if USE_HEAVY_MODEL:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        from peft import PeftModel, PeftConfig

        try:
            peft_cfg = PeftConfig.from_pretrained(ADAPTER_REPO)
            if getattr(peft_cfg, 'base_model_name_or_path', None):
                BASE_MODEL = peft_cfg.base_model_name_or_path
        except Exception:
            pass

        tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=dtype,
                bnb_4bit_use_double_quant=True,
            ),
            torch_dtype=dtype,
            trust_remote_code=True,
            low_cpu_mem_usage=True,
            device_map='auto' if torch.cuda.is_available() else None,
        )
        model = PeftModel.from_pretrained(model, ADAPTER_REPO)
        model.eval()
        print('Model loaded successfully.')
    except Exception as e:
        print('Model load skipped/failure (safe):', e)
else:
    print('Heavy model load skipped by design. Set USE_HEAVY_MODEL=True to load adapter.')

In [ ]:
if model is None or tokenizer is None:
    print('Inference skipped: model/tokenizer not loaded (this is expected with USE_HEAVY_MODEL=False).')
else:
    import torch
    prompt = (
        'You are a CascadeGuard agent. Current failures: POWER_DIST_1, WATER_PUMP_2. '
        'Critical node: HOSP_1. Budget: 6. Return one best next action in JSON.'
    )
    try:
        inputs = tokenizer(prompt, return_tensors='pt')
        if torch.cuda.is_available():
            inputs = {k: v.to('cuda:0') for k, v in inputs.items()}

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=96,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id,
            )
        print(tokenizer.decode(out[0], skip_special_tokens=True))
    except Exception as e:
        print('Inference failed safely:', e)